## DFT vs. MLIPs Energy Comparison

on DFT data for N on Ru slab and NH3 on Ru slab

#### 1. DFT data ingestion

In [9]:
from pathlib import Path
from ase.io import iread, read, write

In [10]:
# load vasp trajectories into ASE Atoms obj.
# metadata: (system type, source dataset, external b-field strength)

def load_frames(path, system, source, bext: float):
    frames = []
    for f in iread(path):
        f.info["system"] = system
        f.info["source"] = source
        f.info["BEXT"] = bext
        frames.append(f)
    
    return frames

def load_frame(path, system, source, bext: float):
    frame = read(path)
    frame.info["system"] = system
    frame.info["source"] = source
    frame.info["BEXT"] = bext

    return frame

In [11]:
# ingest data for MLIP training set

ads = load_frames(
    "dft-data/NH3diss2_BEXT0.1/vasprun.xml",
    system="adsorbed",
    source="NH3diss",
    bext=0.1
)

slab = load_frame(
    "dft-data/NH3surf_BEXT0.1/vasprun.xml",
    system="slab",
    source="NH3surf",
    bext=0.1
)

gas = load_frame(
    "dft-data/NH3mag2_BEXT0.1/vasprun.xml",
    system="adsorbate",
    source="NH3mag2",
    bext=0.1
)

In [12]:
# write full trajectory + single frames for references (for model script handoff below)

write("input_slab_ads.xyz", ads)  # all frames
write("input_slab.xyz", slab)     # single frame
write("input_gas.xyz", gas)       # single frame

#### 2. run MLIPs (MACE, MatterSim, UMA)

In [17]:
# b/c different dependancies, each model is in a separate conda env
import subprocess

r1 = subprocess.run(
    ["/Users/zschwab/miniconda3/envs/mlip-mace/bin/python", "run_mace.py",
    "input_slab_ads.xyz", "input_slab.xyz", "input_gas.xyz"],
    capture_output=True, text=True
)
print(r1.stdout)

r2 = subprocess.run(
    ["/Users/zschwab/miniconda3/envs/mlip-mattersim/bin/python", "run_mattersim.py",
    "input_slab_ads.xyz", "input_slab.xyz", "input_gas.xyz"],
    capture_output=True, text=True
)
print(r2.stdout)

r3 = subprocess.run(
    ["/Users/zschwab/miniconda3/envs/mlip-uma/bin/python", "run_uma.py",
    "input_slab_ads.xyz", "input_slab.xyz", "input_gas.xyz"],
    capture_output=True, text=True
)
print(r3.stdout)

hello from mace!

hello from mattersim!

hello from uma!



In [ ]:
import json, ase.io, pandas as pd

dft_e = ase.io.read("NH3diss2_BEXT0.1/vasprun.xml").get_potential_energy()

mace_e = json.load(open("results_mace.json"))["slab_ads"]["energy"]
ms_e   = json.load(open("results_mattersim.json"))["slab_ads"]["energy"]
uma_e  = json.load(open("results_uma.json"))["slab_ads"]["energy"]

# MAE = mean absolute error
df = pd.DataFrame([
    {"model": "DFT",       "E (eV)": round(dft_e, 4), "MAE vs DFT": 0},
    {"model": "MACE",      "E (eV)": round(mace_e, 4), "MAE vs DFT": round(abs(mace_e - dft_e), 4)},
    {"model": "MatterSim", "E (eV)": round(ms_e, 4),   "MAE vs DFT": round(abs(ms_e - dft_e), 4)},
    {"model": "UMA",       "E (eV)": round(uma_e, 4),  "MAE vs DFT": round(abs(uma_e - dft_e), 4)},
])
df

In [4]:
from fairchem.core import pretrained_mlip, FAIRChemCalculator
from mattersim.forcefield import MatterSimCalculator
from mace.calculators import mace_mp

W0601 16:37:35.861000 84018 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


RuntimeError: operator torchvision::nms does not exist

In [ ]:
# run MACE and save results

atoms = ase.io.read("NH3diss2_BEXT0.1/vasprun.xml")
calc = mace_mp(model="medium", dispersion=False, default_dtype="float32")  # dispersion=false for consistency with other models
atoms.calc = calc

results_mace = {"slab_ads": {"energy": atoms.get_potential_energy(),
                              "forces": atoms.get_forces().tolist()}}
with open("results_mace.json", "w") as f:
    json.dump(results_mace, f)
print("MACE done:", results_mace["slab_ads"]["energy"], "eV")

In [ ]:
# define calculators (UMA, MatterSim, MACE)

predictor = pretrained_mlip.get_predict_unit("uma-s-1p2", device="cpu")  # change to cuda if available

uma_calc = FAIRChemCalculator(
    predictor,
    task_name="oc20",
    seed=None
)

mattersim_calc = MatterSimCalculator()

mace_calc = mace_mp(
    model="medium", 
    dispersion=False,  # dispersion=false for consistency with other models (even though disp. impact on NH3-Ru is nontrivial)
    default_dtype="float32"
)

2026-06-01 16:13:48.025 | INFO     | mattersim.forcefield.potential:from_checkpoint:866 - The pre-trained model is not found locally, attempting to download it from the server.
2026-06-01 16:13:48.031 | INFO     | mattersim.utils.download_utils:download_file:20 - Downloading file from https://raw.githubusercontent.com/microsoft/mattersim/main/pretrained_models/mattersim-v1.0.0-1M.pth to /Users/zschwab/.local/mattersim/pretrained_models/mattersim-v1.0.0-1M.pth
2026-06-01 16:13:49.175 | INFO     | mattersim.utils.download_utils:download_file:26 - File downloaded to /Users/zschwab/.local/mattersim/pretrained_models/mattersim-v1.0.0-1M.pth
2026-06-01 16:13:49.177 | INFO     | mattersim.forcefield.potential:from_checkpoint:873 - Loading the pre-trained mattersim-v1.0.0-1M.pth model
Using Materials Project MACE for MACECalculator with /Users/zschwab/.cache/mace/20231203mace128L1_epoch199model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64

/Users/zschwab/miniconda3/envs/mlip/lib/python3.11/site-packages/mattersim/forcefield/potential.py:892: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  checkpoint = torch.load(load_path, map_location=device)
/Users/zschwab/miniconda3/envs/mlip/lib/python3.11/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


ValueError: too many values to unpack (expected 2)

In [20]:
import inspect
print(inspect.signature(FAIRChemCalculator))

(predict_unit: 'MLIPPredictUnit', task_name: 'UMATask | str | None' = None, seed: 'int | None' = None)


In [5]:
from fairchem.core import pretrained_mlip, FAIRChemCalculator